# 10 — Nonlinear 2D quasi-reversibility (QR): production + fair comparison

**Run All** after notebook 08. Notebook 09 can be run before or afterward; no PINN models are trained here. This notebook implements the **mixed fourth-order quasi-reversibility equations actually printed in the paper** with the nonlinear coefficient $a(w^n)$ lagged at each reverse step. It does not silently substitute a Tikhonov initial-condition optimization.

- Essential condition $w|_{\partial\Omega}=0$; auxiliary field $q$ has **no imposed Dirichlet boundary condition**. This carries the same natural derivative closure as the manuscript.
- The clean and nominal 5% noisy **961 interior sensor vectors** are precisely the files frozen for the three PINN implementations.
- Gamma is chosen from **independent forward-terminal sensor mismatch**, using the declared noise scale (the true initial field is not used for tuning).
- Failed gamma candidates remain in the CSV. No field clipping and no invented results.
- Reconstruction figures require **both QR and an actual fixed-Fourier PINN result**; the final field figure compares **reference, QR, Fourier PINN** (not reference + three PINNs). The other neural results are reported in a comparison table.

The first phase solves 2D mixed $P_1$ finite elements on a 33×33 nodal grid using an implicit lagged-coefficient reverse step. This is a **nonlinear QR extension** of the original mixed FreeFEM formulation, not the original frozen linear-2D QR data.

In [ ]:
from pathlib import Path
import os,sys,json
import numpy as np
import pandas as pd
from scipy.interpolate import RegularGridInterpolator
from IPython.display import display, Image
ROOT=Path.cwd().resolve()
if not (ROOT/'inverse_study').is_dir() and (ROOT/'heat'/'inverse_study').is_dir(): ROOT=ROOT/'heat'
if not (ROOT/'inverse_study').is_dir():raise FileNotFoundError('Open notebook in the heat project directory')
sys.path.insert(0,str(ROOT))
from inverse_study import qr_nonlinear2d as qr, nonlinear2d as n2, nonlinear2d_figures as plots
R=ROOT/'results'/'nonlinear2d'
SPEC=n2.problem_spec()
REFERENCE=n2.load_reference(R/'reference'/'reference_n97_dt0.005.npz')
CONDITIONS={'clean':'noise0_seed0','noisy5':'noise0.05_seed3100'}
# Predeclared pilot grid: deliberately bracket both instability and oversmoothing.
GAMMAS=(1e-5, 1.5e-5, 2e-5, 2.5e-5, 3e-5, 4e-5, 6e-5, 1e-4, 3e-4, 1e-3)
QR_NODES=33
QR_DT=0.02
REPLAY_NODES=33
REPLAY_DT=0.02
RUN_QR_PRODUCTION=os.environ.get('NONLINEAR2D_QR_AUDIT_ONLY','0') != '1'
RUN_SELECTED_REFINEMENT=os.environ.get('NONLINEAR2D_QR_SKIP_REFINEMENT','0') != '1'
SELECTED_NEURAL_FOR_FIELD_FIGURE='fourier_fixed'  # predeclared; NEVER oracle-selected
print('QR production:',RUN_QR_PRODUCTION,'refinement:',RUN_SELECTED_REFINEMENT)
print('candidate solves:', len(CONDITIONS)*len(GAMMAS),'same frozen observation grids as PINNs')

## Mixed finite-element identity and boundary-closure checks

We check the consistent mass matrix, symmetric Laplacian stiffness, and weak-form residual after a test reverse step. This sanity test is independent of the unknown initial-state reconstruction.

In [ ]:
fm=qr.MixedFEM2D(9)
assert np.allclose((fm.mass-fm.mass.T).data,0,atol=1e-12)
assert np.allclose((fm.k_lap-fm.k_lap.T).data,0,atol=1e-12)
assert np.isclose(float(fm.mass.sum()),1.,atol=1e-12)
xx,yy=np.meshgrid(fm.axis,fm.axis,indexing='ij')
trial=np.sin(np.pi*xx)*np.sin(np.pi*yy)
test=fm.run(trial,SPEC,gamma=0.003,dt=0.1)
assert test['max_weak1_residual']<1e-10 and test['max_weak2_residual']<1e-10
assert np.allclose(test['initial'][[0,-1],:],0,atol=1e-12)
assert np.allclose(test['initial'][:,[0,-1]],0,atol=1e-12)
print('Mixed weak-form algebra and boundary enforcement: PASS')

## Complete QR gamma sweep on the actual clean and 5% terminal measurements

Every successful candidate is independently advanced forward with the **nonlinear conservative solver**; this is separate code from the backward mixed FE solver. The first replay uses 33×33 and $\Delta t=0.02$ to screen gamma efficiently. Selected results are then checked on a finer grid. A noisy discrepancy target is $\sqrt{N_{\mathrm{obs}}}\sigma$; clean uses minimum forward mismatch. The selection record indicates whether that target was attained.

In [ ]:
all_rows=[]
selection={}
for condition,tag in CONDITIONS.items():
    obs,meta=qr.load_observation(R/'observations'/tag)
    if meta['experiment']!=SPEC:raise RuntimeError('Physical-law mismatch in frozen observations')
    if RUN_QR_PRODUCTION:
        rows=[]
        for gamma in GAMMAS:
            rec,folder,cache_state=qr.run_or_load_candidate(
                R/'qr_mixed'/tag,obs,meta,SPEC,gamma,
                n=QR_NODES,dt=QR_DT,replay_n=REPLAY_NODES,replay_dt=REPLAY_DT)
            row={'condition':condition,'folder':str(folder.relative_to(ROOT)),
                 'cached_or_computed':cache_state,**rec}
            rows.append(row);all_rows.append(row)
            print(f"{condition:6s} gamma={gamma:9.2g} {rec['status']:9s} ",
                  f"d_obs={rec.get('sensor_discrepancy_relative',float('nan')):.5g}",
                  rec.get('reason','')[:80])
        accepted=qr.choose_by_discrepancy(rows,meta)
        accepted['condition']=condition
        accepted['observation_tag']=tag
        selection[condition]=accepted
        d=R/'qr_mixed'/tag
        (d/'selected_gamma.json').write_text(json.dumps(accepted,indent=2))
    else:
        print('Audit-only: QR computation skipped for',condition)
report=R/'report';report.mkdir(parents=True,exist_ok=True)
if all_rows:
    sweep=pd.DataFrame(all_rows)
    sweep.to_csv(report/'nonlinear2d_qr_gamma_sweep.csv',index=False)
    display(sweep[['condition','gamma','status','sensor_discrepancy_relative','min_temperature','max_temperature','cached_or_computed']])
    print('Non-oracle selected QR candidates:')
    display(pd.DataFrame(selection.values()))

## Post-selection metrics and independent refinement (without retraining)

**Only here** is the prescribed initial state evaluated. Refinement replays the fixed selected reconstruction on 65×65 with a smaller time step. It also reruns **the same selected gamma** on a finer mixed-FE mesh to document discretization sensitivity; that check does not silently replace the selected primary QR reconstruction or retune gamma.

In [ ]:
comparison=[]
for condition,accepted in selection.items():
    obs,meta=qr.load_observation(R/'observations'/accepted['observation_tag'])
    row=next(z for z in all_rows if z.get('identifier')==accepted['selected_run'] and z['condition']==condition)
    out=ROOT/row['folder']
    with np.load(out/'reconstruction.npz') as z:
        selected_initial=z['initial'].copy()
        qraxis=z['axis'].copy()
    fm=qr.MixedFEM2D(QR_NODES)
    truth_metrics=qr.evaluate_with_truth_only_after_selection(fm,{'initial':selected_initial},REFERENCE)
    record={'condition':condition,'method':'nonlinear_QR_mixed_P1',
            'gamma':accepted['selected_gamma'],'observation_tag':accepted['observation_tag'],
            'run_folder':row['folder'],'independent_sensor_relative_discrepancy':row['sensor_discrepancy_relative'],
            'discrepancy_target_attained':accepted['target_attained_within_20pct'],
            **truth_metrics}
    if RUN_SELECTED_REFINEMENT:
        fine=qr.sensor_fit_from_replay(selected_initial,fm,obs,SPEC,replay_n=65,replay_dt=0.01)
        record['fine_replay_sensor_relative_discrepancy']=fine['sensor_discrepancy_relative']
        np.savez_compressed(out/'selected_fine_forward_replay.npz',
                            axis=fine['replay']['axis'],initial=fine['projected_initial'],
                            terminal=fine['replay']['terminal'])
        # Spatial/time QR refinement of the selected fixed gamma, not a new tuned candidate.
        fine_fem=qr.MixedFEM2D(65)
        ft,origin=fine_fem.sample_sensor_values(obs)
        try:
            fine_qr=fine_fem.run(ft,SPEC,accepted['selected_gamma'],dt=0.01)
            x,y=np.meshgrid(fine_fem.axis,fine_fem.axis,indexing='ij')
            interp=RegularGridInterpolator((qraxis,qraxis),selected_initial)(np.column_stack((x.ravel(),y.ravel()))).reshape(65,65)
            record['qr_mesh_time_refinement_relative_change']=n2.relative_l2(fine_qr['initial'],interp)
            record['qr_fine_initial_relative_l2']=qr.evaluate_with_truth_only_after_selection(fine_fem,fine_qr,REFERENCE)['initial_relative_l2']
            np.savez_compressed(out/'selected_qr_fine_refinement.npz',axis=fine_fem.axis,initial=fine_qr['initial'],
                                qr_fine_gamma=accepted['selected_gamma'],qr_fine_dt=0.01)
        except Exception as exc:
            record['qr_mesh_time_refinement_failed']=f'{type(exc).__name__}: {exc}'
            print('QR fine mesh check failed (recorded, not replaced):',record['qr_mesh_time_refinement_failed'])
    comparison.append(record)
if comparison:
    result=pd.DataFrame(comparison)
    result.to_csv(report/'nonlinear2d_qr_selected_results.csv',index=False)
    display(result)
else:print('No selected QR candidates yet; no fabricated output.')

## Actual QR–PINN comparison, in the author's original field style

No 4-panel ``reference + three PINNs`` field plot is produced. Once notebook 09 has completed, this notebook creates one figure per condition with **reference, mixed nonlinear QR, and predeclared fixed-weight Fourier PINN** using the author's jet/colorbar/contour conventions. All three neural methods are included numerically in the comparison CSV when their real results are available. If the neural jobs are unfinished, figures are not generated.

In [ ]:
rows=[]
for record in comparison:
    rows.append({'condition':record['condition'],'method':'QR (mixed nonlinear)',
                 'initial_relative_l2':record['initial_relative_l2'],
                 'independent_sensor_relative_discrepancy':record['fine_replay_sensor_relative_discrepancy']
                    if 'fine_replay_sensor_relative_discrepancy' in record else record['independent_sensor_relative_discrepancy'],
                 'source':record['run_folder']})
models={}
for f in sorted((R/'neural').glob('*/status.json')):
    status=json.loads(f.read_text())
    if status.get('status')!='completed' or status.get('profile')!='production':continue
    folder=f.parent
    cfg=json.loads((folder/'config.json').read_text())
    if cfg.get('problem',{}).get('law')!='sigmoid20_2d':continue
    rid=cfg['config'].get('run_id','')
    condition=next((cond for cond in CONDITIONS if f'strong2d_{cond}_' in rid),None)
    if condition is None:continue
    method=cfg['config']['method']
    evaluation=folder/'evaluation_nonlinear2d.json'
    if not evaluation.exists():
        print('Model trained but evaluation missing:',folder,'; run notebook 09')
        continue
    ev=json.loads(evaluation.read_text())
    models[condition,method]=folder
    rows.append({'condition':condition,'method':method,
                 'initial_relative_l2':ev['initial_relative_l2'],
                 'independent_sensor_relative_discrepancy':ev['independent_sensor_relative_discrepancy'],
                 'source':str(folder.relative_to(ROOT))})
if rows:
    df=pd.DataFrame(rows).sort_values(['condition','method'])
    df.to_csv(report/'nonlinear2d_qr_pinn_comparison.csv',index=False)
    display(df)
for condition,accepted in selection.items():
    key=(condition,SELECTED_NEURAL_FOR_FIELD_FIGURE)
    if key not in models:
        print(f'{condition}: no completed {SELECTED_NEURAL_FOR_FIELD_FIGURE} prediction; comparative field plot not generated')
        continue
    row=next(r for r in comparison if r['condition']==condition)
    with np.load(ROOT/row['run_folder']/'reconstruction.npz') as z:
        qa=z['axis'].copy();qi=z['initial'].copy()
    with np.load(models[key]/'predictions.npz') as z:
        xev=z['X_eval'];nu=z['theta_initial']
    vv,xx,yy=n2.grid(65)
    expected=np.column_stack((xx.ravel(),yy.ravel()))
    if not np.allclose(xev,expected):raise RuntimeError('Neural evaluation grid does not match declared comparison grid')
    q65=RegularGridInterpolator((qa,qa),qi,bounds_error=True)(expected).reshape(65,65)
    base=R/'figures'/f'nonlinear2d_qr_vs_fourier_{condition}_original_style'
    plots.plot_qr_vs_fourier({'initial':n2.initial(xx,yy)},q65,nu,base)
    print('Comparative field figure saved:',base.with_suffix('.pdf'))
    display(Image(filename=str(base.with_suffix('.png'))))
print('Finished. Refer to results/nonlinear2d/report/*.csv for measured outcomes.')